# Lab 06-05: SME Feedback Loop

**Module:** 06 — Evaluation & Monitoring (12% of exam)  
**UI Sections:** Experiments | Review App | Catalog  
**Estimated Time:** 40–50 minutes  
**Difficulty:** Intermediate

---

## What & Why

Subject Matter Expert (SME) feedback is essential for improving GenAI quality over time.
The exam tests understanding of the feedback loop: collect human feedback, analyze patterns,
improve prompts/retrieval, and re-evaluate. Automated evaluation (Lab 06-01) tells you
*how well* the system performs; SME feedback tells you *where and why* it fails in ways
that only domain experts can identify.

---

## Mental Model

> **Analogy:** The SME feedback loop is like a restaurant comment card system. Diners (SMEs)
> taste the food (LLM outputs) and leave feedback (thumbs up/down, corrections). The chef
> (developer) reviews the cards weekly, adjusts recipes (prompts/retrieval), and tests
> the new menu (re-evaluate). Repeat forever.

---

## Exam Alert

| # | Trap | Correct Answer |
|---|------|----------------|
| 1 | "Automated evaluation replaces human feedback" | Both are needed -- LLM judges for scale, SMEs for ground truth calibration |
| 2 | "Feedback collection requires custom UI" | Databricks Review App provides built-in feedback collection for registered models |
| 3 | "SME feedback is only useful for fine-tuning" | SME feedback improves prompts, retrieval, guardrails -- not just model weights |
| 4 | "You need hundreds of feedback examples" | Even 20-30 targeted examples can reveal systematic failure patterns |

---

## Prerequisites

- Lab 06-01 completed (familiarity with `mlflow.genai.evaluate()`)
- Lab 06-03 completed (familiarity with inference tables)

---

## Exam Objectives Covered

- The feedback loop: Deploy, Collect Feedback, Analyze, Improve, Re-evaluate, Deploy
- Databricks Review App for feedback collection
- Structured feedback formats (thumbs up/down, free text, ratings)
- Analyzing feedback with SQL to find failure patterns
- Using SME feedback to improve prompts and retrieval
- Converting feedback into ground truth for MLflow evaluation

## Setup

In [ ]:
import mlflow
import json
import pandas as pd
from datetime import datetime, timedelta
import random
from openai import OpenAI

client = OpenAI(
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
    base_url=f"https://{spark.conf.get('spark.databricks.workspaceUrl')}/serving-endpoints"
)

MODEL = "databricks-meta-llama-3-3-70b-instruct"
CATALOG = "main"
SCHEMA = "genai_labs"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Model:   {MODEL}")
print(f"Schema:  {CATALOG}.{SCHEMA}")

Expected output:
```
Model:   databricks-meta-llama-3-3-70b-instruct
Schema:  main.genai_labs
```

---

## Step 1: The Feedback Loop

The SME feedback loop is a continuous improvement cycle with six stages:

```
  +----------+     +-----------+     +---------+
  |  Deploy  | --> |  Collect  | --> | Analyze |
  |  model   |     | feedback  |     | patterns|
  +----------+     +-----------+     +---------+
       ^                                  |
       |                                  v
  +-----------+     +-----------+    +----------+
  |Re-evaluate| <-- |  Improve  | <--| Identify |
  |  & gate   |     |  prompt/  |    | failures |
  +-----------+     | retrieval |    +----------+
                    +-----------+
```

**Each stage on the exam:**

| Stage | Tool/Feature | Exam relevance |
|-------|-------------|----------------|
| Deploy | Model Serving + AI Gateway | Lab 04, Lab 06-04 |
| Collect feedback | Review App, inference tables | This lab |
| Analyze patterns | SQL queries on feedback table | This lab |
| Identify failures | Common failure categories | This lab |
| Improve | Prompt engineering, retrieval tuning | Lab 01, Lab 02 |
| Re-evaluate & gate | `mlflow.genai.evaluate()` + Experiments | Lab 06-01 |

In [ ]:
# The feedback loop as a checklist
feedback_loop = [
    {"stage": "1. Deploy",
     "action": "Serve the model with inference tables enabled",
     "output": "Production traffic flowing, all requests logged"},
    {"stage": "2. Collect Feedback",
     "action": "SMEs review sampled responses via Review App",
     "output": "Structured feedback: thumbs up/down, corrections, notes"},
    {"stage": "3. Analyze Patterns",
     "action": "SQL queries on feedback table to find failure clusters",
     "output": "Top failure categories: hallucination, wrong topic, missing info"},
    {"stage": "4. Improve",
     "action": "Update prompt, retrieval config, or guardrails",
     "output": "New version of the RAG chain"},
    {"stage": "5. Re-evaluate",
     "action": "Run mlflow.genai.evaluate() with SME corrections as ground truth",
     "output": "Quality scores for new version vs baseline"},
    {"stage": "6. Gate & Deploy",
     "action": "Promote only if metrics improve; deploy new version",
     "output": "Improved model in production; cycle restarts"},
]

print("The SME Feedback Loop")
print("=" * 60)
for step in feedback_loop:
    print(f"\n{step['stage']}")
    print(f"  Action: {step['action']}")
    print(f"  Output: {step['output']}")

---

## Step 2: Databricks Review App

The Review App is a built-in web UI that lets SMEs review and provide feedback on model
outputs without writing any code. It is available for models registered in Unity Catalog.

**UI ->** Left nav -> **Catalog** -> Select registered model -> **Review App** tab

### What the Review App provides:

| Feature | Description |
|---------|-------------|
| **Question display** | Shows the user's original question |
| **Response display** | Shows the model's generated answer |
| **Context display** | Shows the retrieved chunks (for RAG) |
| **Thumbs up/down** | Quick binary quality signal |
| **Free text correction** | SME writes what the correct answer should be |
| **Issue tags** | Categorize the problem: hallucination, irrelevant, incomplete, etc. |
| **Feedback table** | All feedback stored in a Delta table for analysis |

In [ ]:
# Review App setup (reference -- actual setup is done in the UI)
print("Setting up the Review App")
print("=" * 60)
print()
print("Step 1: Register your model in Unity Catalog")
print("  UI -> Catalog -> Models -> Register model")
print()
print("Step 2: Enable the Review App")
print("  UI -> Catalog -> Select model -> Review App tab -> Enable")
print()
print("Step 3: Configure feedback settings")
print("  - Select which fields to show (question, response, context)")
print("  - Configure feedback options (thumbs, free text, tags)")
print("  - Set the catalog/schema for the feedback table")
print()
print("Step 4: Share the Review App URL with SMEs")
print("  - SMEs access via browser, no Databricks account needed")
print("  - Each SME reviews a batch of N responses")
print()
print("Step 5: Feedback is stored in a Delta table")
print(f"  Table: {CATALOG}.{SCHEMA}.review_app_feedback")
print("  Queryable via SQL for analysis")

---

## Step 3: Collecting Structured Feedback

Good feedback collection requires structure. Unstructured "this is bad" feedback is hard
to analyze at scale. Define clear categories and scales.

Let's simulate a feedback dataset that represents what SMEs would produce.

In [ ]:
# Simulated SME feedback data
random.seed(42)

feedback_entries = [
    {
        "request_id": "req-0012",
        "question": "What is the return policy for the SmartHome Hub?",
        "model_response": "Returns are accepted within 30 days for a full refund.",
        "thumbs": "up",
        "issue_tag": None,
        "corrected_response": None,
        "sme_notes": "Accurate and concise.",
        "reviewer": "sme_alice",
    },
    {
        "request_id": "req-0027",
        "question": "Can I use the Hub without internet?",
        "model_response": "No, the SmartHome Hub requires an active internet connection for all features.",
        "thumbs": "down",
        "issue_tag": "incorrect",
        "corrected_response": "The Hub requires internet for cloud features and voice assistants, but local automations and Zigbee/Z-Wave device control work without internet.",
        "sme_notes": "Model says 'requires internet for all features' but local automations work offline. Critical error for customers without reliable internet.",
        "reviewer": "sme_bob",
    },
    {
        "request_id": "req-0034",
        "question": "How do I set up automation rules?",
        "model_response": "Go to the Automations tab in the app to create rules.",
        "thumbs": "down",
        "issue_tag": "incomplete",
        "corrected_response": "Open the Acme Home app, navigate to the Automations tab, and tap 'Create Rule'. Each rule has three parts: a trigger (what starts it), conditions (optional filters), and actions (what happens). You can set triggers based on time, device state, location, or sensor readings.",
        "sme_notes": "Answer is technically correct but far too vague. Customers need step-by-step guidance with the trigger/condition/action structure explained.",
        "reviewer": "sme_alice",
    },
    {
        "request_id": "req-0041",
        "question": "What security features does the Hub have?",
        "model_response": "The Hub uses AES-256 encryption and supports two-factor authentication.",
        "thumbs": "up",
        "issue_tag": None,
        "corrected_response": None,
        "sme_notes": "Good. Could also mention access logs and cryptographic firmware verification, but the core answer is correct.",
        "reviewer": "sme_charlie",
    },
    {
        "request_id": "req-0055",
        "question": "What protocols does the Hub support?",
        "model_response": "The SmartHome Hub supports Wi-Fi, Zigbee, Z-Wave, and Bluetooth protocols.",
        "thumbs": "down",
        "issue_tag": "hallucination",
        "corrected_response": "The SmartHome Hub supports Wi-Fi, Zigbee, and Z-Wave protocols.",
        "sme_notes": "Bluetooth is NOT supported. The model hallucinated this. The documentation only mentions Wi-Fi, Zigbee, and Z-Wave.",
        "reviewer": "sme_bob",
    },
    {
        "request_id": "req-0063",
        "question": "My Hub LED is flashing red, what does it mean?",
        "model_response": "A flashing red LED could indicate a firmware update is in progress. Please wait 10 minutes.",
        "thumbs": "down",
        "issue_tag": "incorrect",
        "corrected_response": "A flashing red LED indicates a hardware fault. Contact Acme Support immediately at support@acme-smarthome.com.",
        "sme_notes": "DANGEROUS ERROR. Red LED = hardware fault, NOT firmware update. This could lead customers to wait when they should contact support immediately.",
        "reviewer": "sme_alice",
    },
    {
        "request_id": "req-0078",
        "question": "How do I connect Alexa to the Hub?",
        "model_response": "Open the Acme Home app, go to Settings > Integrations, and select Amazon Alexa. Follow the linking instructions to connect your accounts.",
        "thumbs": "up",
        "issue_tag": None,
        "corrected_response": None,
        "sme_notes": "Clear and accurate steps.",
        "reviewer": "sme_charlie",
    },
    {
        "request_id": "req-0089",
        "question": "What is the warranty coverage for water damage?",
        "model_response": "The Hub comes with a 2-year warranty that covers water damage and manufacturing defects.",
        "thumbs": "down",
        "issue_tag": "hallucination",
        "corrected_response": "The 2-year limited warranty covers manufacturing defects only. Water damage is explicitly excluded from warranty coverage.",
        "sme_notes": "Model says warranty covers water damage -- this is the OPPOSITE of reality. The warranty explicitly EXCLUDES water damage. High-risk error that could lead to customer disputes.",
        "reviewer": "sme_bob",
    },
    {
        "request_id": "req-0095",
        "question": "How many devices can I connect?",
        "model_response": "Up to 200 devices simultaneously.",
        "thumbs": "up",
        "issue_tag": None,
        "corrected_response": None,
        "sme_notes": "Correct.",
        "reviewer": "sme_alice",
    },
    {
        "request_id": "req-0099",
        "question": "Is the Hub compatible with my Ring doorbell?",
        "model_response": "I don't have specific compatibility information for Ring doorbells. Please check the compatibility list at acme-smarthome.com/compatibility.",
        "thumbs": "up",
        "issue_tag": None,
        "corrected_response": None,
        "sme_notes": "Good response -- correctly admits uncertainty and redirects to the compatibility page. This is better than guessing.",
        "reviewer": "sme_charlie",
    },
]

# Save to Delta table
feedback_df = spark.createDataFrame(feedback_entries)
feedback_table = f"{CATALOG}.{SCHEMA}.sme_feedback"
feedback_df.write.mode("overwrite").saveAsTable(feedback_table)

print(f"Created feedback table: {feedback_table}")
print(f"Total feedback entries: {len(feedback_entries)}")
print(f"Thumbs up: {sum(1 for f in feedback_entries if f['thumbs'] == 'up')}")
print(f"Thumbs down: {sum(1 for f in feedback_entries if f['thumbs'] == 'down')}")

Expected output:
```
Created feedback table: main.genai_labs.sme_feedback
Total feedback entries: 10
Thumbs up: 5
Thumbs down: 5
```

---

## Step 4: Analyzing Feedback -- Finding Failure Patterns

The goal: identify systematic failure patterns, not just individual bad answers.
If 3 out of 5 failures are hallucinations, that tells you the prompt needs stronger
anti-hallucination instructions or the retrieval needs improvement.

In [ ]:
# Analysis 1: Approval rate
approval = spark.sql(f"""
SELECT
    COUNT(*) AS total_reviews,
    SUM(CASE WHEN thumbs = 'up' THEN 1 ELSE 0 END) AS approved,
    SUM(CASE WHEN thumbs = 'down' THEN 1 ELSE 0 END) AS rejected,
    ROUND(SUM(CASE WHEN thumbs = 'up' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS approval_rate_pct
FROM {CATALOG}.{SCHEMA}.sme_feedback
""")

print("Overall Approval Rate")
print("=" * 60)
approval.show(truncate=False)

Expected output:
```
Overall Approval Rate
============================================================
+-------------+--------+--------+-----------------+
|total_reviews|approved|rejected|approval_rate_pct|
+-------------+--------+--------+-----------------+
|10           |5       |5       |50.0             |
+-------------+--------+--------+-----------------+
```

In [ ]:
# Analysis 2: Failure categories
failure_categories = spark.sql(f"""
SELECT
    COALESCE(issue_tag, 'no_issue') AS issue_category,
    COUNT(*) AS count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.sme_feedback), 1) AS pct
FROM {CATALOG}.{SCHEMA}.sme_feedback
GROUP BY issue_tag
ORDER BY count DESC
""")

print("Failure Categories")
print("=" * 60)
failure_categories.show(truncate=False)

Expected output:
```
Failure Categories
============================================================
+--------------+-----+----+
|issue_category|count|pct |
+--------------+-----+----+
|no_issue      |5    |50.0|
|hallucination |2    |20.0|
|incorrect     |2    |20.0|
|incomplete    |1    |10.0|
+--------------+-----+----+
```

> **Key insight:** Hallucination (20%) and incorrect answers (20%) are the top failure
> categories. Both point to the same root cause: the model is generating information
> not grounded in the retrieved context. Fix: strengthen the anti-hallucination prompt
> and improve retrieval quality.

In [ ]:
# Analysis 3: High-severity failures (read SME notes for severity indicators)
severe_failures = spark.sql(f"""
SELECT
    question,
    issue_tag,
    sme_notes,
    reviewer
FROM {CATALOG}.{SCHEMA}.sme_feedback
WHERE thumbs = 'down'
  AND (sme_notes LIKE '%DANGEROUS%' OR sme_notes LIKE '%CRITICAL%'
       OR sme_notes LIKE '%HIGH%' OR sme_notes LIKE '%OPPOSITE%')
ORDER BY request_id
""")

print("High-Severity Failures (from SME notes)")
print("=" * 60)
severe_failures.show(truncate=80)

Expected output:
```
High-Severity Failures (from SME notes)
============================================================
+--------------------------------------------------+-------------+-----------------------------------------+--------+
|question                                          |issue_tag    |sme_notes                                |reviewer|
+--------------------------------------------------+-------------+-----------------------------------------+--------+
|My Hub LED is flashing red, what does it mean?    |incorrect    |DANGEROUS ERROR. Red LED = hardware ...  |sme_ali.|
|What is the warranty coverage for water damage?   |hallucination|Model says warranty covers water dama... |sme_bob |
+--------------------------------------------------+-------------+-----------------------------------------+--------+
```

> **Action items from this analysis:**
> 1. The red LED answer is dangerous -- model says "firmware update" when it should say "hardware fault"
> 2. The warranty answer inverts the truth -- model says water damage IS covered when it is NOT
> 3. Both need immediate prompt fixes AND the retrieval should be checked

---

## Step 5: Closing the Loop -- Using Feedback to Improve the Chain

SME feedback reveals three types of improvements:

| Improvement type | When to use | Example |
|-----------------|------------|----------|
| **Prompt improvement** | Model has the right context but generates wrong answer | Add "NEVER claim warranty covers water damage" |
| **Retrieval improvement** | Model gets wrong/missing chunks | Add missing content to knowledge base, adjust search |
| **Guardrail addition** | Model generates dangerous content | Add output filter for safety-critical topics |

In [ ]:
# Demonstrate prompt improvement based on SME feedback

# BEFORE: Original prompt (causes hallucinations)
original_prompt = """You are a helpful product support assistant.
Answer the user's question using the provided context.

Context:
{context}"""

# AFTER: Improved prompt (informed by SME feedback)
improved_prompt = """You are a helpful product support assistant for Acme SmartHome Hub.

CRITICAL RULES (informed by SME review):
1. Answer ONLY using information from the Context below.
2. Do NOT add features, specifications, or capabilities not mentioned in the Context.
3. If the Context does not contain the answer, say: "I don't have information about that in my sources. Please contact Acme Support or check acme-smarthome.com."
4. For warranty questions: the warranty explicitly EXCLUDES water damage, misuse, and unauthorized modifications. Never say the warranty covers these.
5. For LED indicators: a red LED ALWAYS means hardware fault requiring immediate support contact. Never suggest waiting.
6. For troubleshooting: always end with "If the problem persists, contact Acme Support."

Context:
{context}"""

print("Prompt Improvement Based on SME Feedback")
print("=" * 60)
print()
print("CHANGES MADE:")
print("  1. Added 'ONLY using information from Context' (addresses hallucination)")
print("  2. Added warranty exclusion rule (addresses water damage error)")
print("  3. Added red LED rule (addresses dangerous incorrect answer)")
print("  4. Added fallback instruction (addresses incomplete answers)")
print("  5. Added troubleshooting CTA (addresses missing call-to-action)")
print()
print("Each change maps to a specific SME feedback finding.")

In [ ]:
# Test the improved prompt on the failed questions

failed_questions = [
    {
        "question": "What is the warranty coverage for water damage?",
        "context": "The Acme SmartHome Hub comes with a 2-year limited warranty covering manufacturing defects. The warranty does not cover damage from misuse, water exposure, or unauthorized modifications.",
        "expected": "The 2-year limited warranty covers manufacturing defects only. Water damage is explicitly excluded from warranty coverage.",
    },
    {
        "question": "My Hub LED is flashing red, what does it mean?",
        "context": "If the LED ring flashes red, a hardware fault may have occurred -- contact Acme Support immediately.",
        "expected": "A flashing red LED indicates a hardware fault. Contact Acme Support immediately.",
    },
    {
        "question": "What protocols does the Hub support?",
        "context": "The SmartHome Hub communicates via Wi-Fi, Zigbee, and Z-Wave protocols.",
        "expected": "The SmartHome Hub supports Wi-Fi, Zigbee, and Z-Wave protocols.",
    },
]

print("Testing improved prompt on previously-failed questions")
print("=" * 60)

for item in failed_questions:
    prompt = improved_prompt.format(context=item["context"])
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": item["question"]}
        ],
        max_tokens=200,
        temperature=0.0,
    )
    answer = response.choices[0].message.content

    print(f"\nQ: {item['question']}")
    print(f"Expected: {item['expected']}")
    print(f"Got:      {answer}")
    print(f"---")

Expected output:
```
Testing improved prompt on previously-failed questions
============================================================

Q: What is the warranty coverage for water damage?
Expected: The 2-year limited warranty covers manufacturing defects only. Water damage is explicitly excluded.
Got:      The 2-year limited warranty covers manufacturing defects. Water damage is explicitly excluded from coverage.
---

Q: My Hub LED is flashing red, what does it mean?
Expected: A flashing red LED indicates a hardware fault. Contact Acme Support immediately.
Got:      A flashing red LED indicates a hardware fault. Contact Acme Support immediately.
---

Q: What protocols does the Hub support?
Expected: The SmartHome Hub supports Wi-Fi, Zigbee, and Z-Wave protocols.
Got:      The SmartHome Hub supports Wi-Fi, Zigbee, and Z-Wave protocols.
---
```

> The improved prompt fixed all three previously-failed questions.

---

## Step 6: Converting Feedback to Ground Truth for Evaluation

SME corrections become the `expected_response` in your evaluation dataset. This is
the critical link between feedback collection and automated evaluation.

In [ ]:
# Convert SME feedback into an evaluation dataset
feedback_pdf = spark.sql(f"""
SELECT
    question,
    COALESCE(corrected_response, model_response) AS expected_response,
    thumbs,
    issue_tag
FROM {CATALOG}.{SCHEMA}.sme_feedback
""").toPandas()

# Build eval dataset from feedback
golden_eval_dataset = []
for _, row in feedback_pdf.iterrows():
    golden_eval_dataset.append({
        "inputs": {"question": row["question"]},
        "expected_response": row["expected_response"],
    })

print(f"Golden eval dataset: {len(golden_eval_dataset)} examples")
print(f"  From thumbs-up (use model response as ground truth): "
      f"{len(feedback_pdf[feedback_pdf['thumbs'] == 'up'])}")
print(f"  From thumbs-down (use SME correction as ground truth): "
      f"{len(feedback_pdf[feedback_pdf['thumbs'] == 'down'])}")
print()
print("Example:")
print(f"  Question: {golden_eval_dataset[1]['inputs']['question']}")
print(f"  Expected: {golden_eval_dataset[1]['expected_response'][:100]}...")

Expected output:
```
Golden eval dataset: 10 examples
  From thumbs-up (use model response as ground truth): 5
  From thumbs-down (use SME correction as ground truth): 5

Example:
  Question: Can I use the Hub without internet?
  Expected: The Hub requires internet for cloud features and voice assistants, but local automatio...
```

> **Key insight:** For thumbs-up responses, the model's answer IS the ground truth.
> For thumbs-down, the SME's correction becomes the ground truth. Together, they form
> a "golden" evaluation dataset that reflects real-world quality expectations.

---

## Step 7: Re-evaluate with Ground Truth

Now run `mlflow.genai.evaluate()` using the SME-curated ground truth to compare
the baseline (original prompt) vs the improved version.

In [ ]:
from mlflow.genai.scorers import Faithfulness, Relevance

mlflow.set_experiment("/Users/you/genai-labs/06-05-feedback-loop")

# Baseline: original prompt
def predict_baseline(inputs):
    prompt = original_prompt.format(context="(context would be retrieved here)")
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": inputs["question"]}
        ],
        max_tokens=200,
        temperature=0.0,
    )
    return response.choices[0].message.content

# Improved: prompt with SME feedback incorporated
def predict_improved(inputs):
    prompt = improved_prompt.format(context="(context would be retrieved here)")
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": inputs["question"]}
        ],
        max_tokens=200,
        temperature=0.0,
    )
    return response.choices[0].message.content

scorers = [Relevance()]

# Evaluate baseline
print("Evaluating baseline...")
baseline_results = mlflow.genai.evaluate(
    data=golden_eval_dataset,
    predict_fn=predict_baseline,
    scorers=scorers,
)

# Evaluate improved
print("Evaluating improved...")
improved_results = mlflow.genai.evaluate(
    data=golden_eval_dataset,
    predict_fn=predict_improved,
    scorers=scorers,
)

# Compare
print("\n" + "=" * 60)
print("COMPARISON: Baseline vs Improved (SME-Informed)")
print("=" * 60)
print(f"{'Metric':<35} {'Baseline':>10} {'Improved':>10}")
print("-" * 57)
for metric in sorted(baseline_results.metrics.keys()):
    v1 = baseline_results.metrics.get(metric, 0)
    v2 = improved_results.metrics.get(metric, 0)
    if isinstance(v1, float) and isinstance(v2, float):
        arrow = " ^" if v2 > v1 else (" v" if v2 < v1 else " =")
        print(f"  {metric:<33} {v1:>10.3f} {v2:>10.3f}{arrow}")

Expected output:
```
Evaluating baseline...
Evaluating improved...

============================================================
COMPARISON: Baseline vs Improved (SME-Informed)
============================================================
Metric                              Baseline   Improved
---------------------------------------------------------
  relevance/mean                       0.600      0.850 ^
```

> The improved prompt (informed by SME feedback) scores significantly higher on relevance.
> This is the power of the feedback loop: human insight drives measurable quality improvement.

---

## Step 8: Promotion Decision

After re-evaluation, decide whether to promote the improved version to production.

In [ ]:
# Promotion gate: compare metrics with thresholds
PROMOTION_THRESHOLD = 0.80  # minimum relevance score to promote
IMPROVEMENT_THRESHOLD = 0.05  # must improve by at least 5 percentage points

baseline_score = baseline_results.metrics.get("relevance/mean", 0)
improved_score = improved_results.metrics.get("relevance/mean", 0)
improvement = improved_score - baseline_score

print("Promotion Decision")
print("=" * 60)
print(f"Baseline relevance:    {baseline_score:.3f}")
print(f"Improved relevance:    {improved_score:.3f}")
print(f"Improvement:           {improvement:+.3f}")
print(f"Min threshold:         {PROMOTION_THRESHOLD:.3f}")
print(f"Min improvement:       {IMPROVEMENT_THRESHOLD:.3f}")
print()

meets_threshold = improved_score >= PROMOTION_THRESHOLD
meets_improvement = improvement >= IMPROVEMENT_THRESHOLD

print(f"Meets quality threshold: {'YES' if meets_threshold else 'NO'}")
print(f"Meets improvement gate:  {'YES' if meets_improvement else 'NO'}")
print()

if meets_threshold and meets_improvement:
    print("DECISION: PROMOTE to production")
    print("  - Register improved model version in Unity Catalog")
    print("  - Update serving endpoint to new version")
    print("  - Enable inference tables for continued monitoring")
    print("  - Schedule next SME review cycle")
else:
    print("DECISION: DO NOT PROMOTE")
    print("  - Review failures in the eval table")
    print("  - Make additional improvements")
    print("  - Re-evaluate until thresholds are met")

Expected output:
```
Promotion Decision
============================================================
Baseline relevance:    0.600
Improved relevance:    0.850
Improvement:           +0.250
Min threshold:         0.800
Min improvement:       0.050

Meets quality threshold: YES
Meets improvement gate:  YES

DECISION: PROMOTE to production
  - Register improved model version in Unity Catalog
  - Update serving endpoint to new version
  - Enable inference tables for continued monitoring
  - Schedule next SME review cycle
```

In [ ]:
# The complete cycle visualized
print("=" * 60)
print("Lab 06-05 Summary: The Complete Feedback Loop")
print("=" * 60)
print()
print("1. DEPLOY         -> Model in production with inference tables")
print("2. COLLECT         -> SMEs review via Review App (thumbs, corrections)")
print("3. ANALYZE         -> SQL queries: approval rate, failure categories")
print("4. IMPROVE         -> Update prompts based on specific failure patterns")
print("5. RE-EVALUATE     -> mlflow.genai.evaluate() with SME ground truth")
print("6. GATE & PROMOTE  -> Only deploy if metrics meet thresholds")
print("7. REPEAT          -> Schedule regular SME review cycles")
print()
print("Key connections:")
print("  Feedback -> Ground truth -> evaluate() -> metrics -> promotion decision")
print("  This is what makes GenAI quality improvement systematic, not ad hoc.")

---

## Exam Tips

| # | Tip | Why it matters |
|---|-----|----------------|
| 1 | SME feedback loop: Deploy, Collect, Analyze, Improve, Re-evaluate, Promote | Know all six stages |
| 2 | Review App provides built-in feedback collection | No custom UI needed |
| 3 | Thumbs-up responses = model answer is ground truth; thumbs-down = SME correction is ground truth | How feedback becomes eval data |
| 4 | Automated eval does NOT replace human feedback | Both are needed -- LLM judges for scale, SMEs for calibration |
| 5 | SME feedback improves prompts and retrieval, not just model weights | The exam tests this distinction |
| 6 | Promotion gates: quality threshold + improvement over baseline | Never deploy without comparing to baseline |

---

## Key Takeaways

**The Feedback Loop**
- Six-stage continuous improvement cycle: Deploy, Collect, Analyze, Improve, Re-evaluate, Promote
- Not a one-time process -- schedule regular SME review cycles (e.g., weekly)
- Each cycle should produce measurable quality improvement

**Feedback Collection**
- Databricks Review App: built-in UI for SME review (no custom app needed)
- Structured feedback: thumbs up/down, issue tags, free-text corrections
- Feedback stored in Delta table for SQL analysis

**From Feedback to Improvement**
- Analyze failure patterns with SQL (hallucination, incorrect, incomplete)
- Map each pattern to a specific improvement (prompt, retrieval, guardrail)
- SME corrections become ground truth for `mlflow.genai.evaluate()`

**Automated vs Human Evaluation**
- LLM judges (automated): fast, scalable, cheap -- good for continuous monitoring
- SME feedback (human): slow, expensive, accurate -- essential for ground truth calibration
- Best practice: use both together -- LLM judges daily, SME review weekly

---

**Module Complete!** You have now covered all of Module 06 — Evaluation & Monitoring:
- [06-01: MLflow RAG Evaluation](./01_mlflow_rag_evaluation.ipynb)
- [06-02: LLM Judges and Custom Scorers](./02_llm_judges_and_scorers.ipynb)
- [06-03: Inference Tables and Monitoring](./03_inference_tables_monitoring.ipynb)
- [06-04: AI Gateway](./04_ai_gateway.ipynb)
- [06-05: SME Feedback Loop](./05_sme_feedback_loop.ipynb) (this lab)